In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from collections import defaultdict
import re

## Configuración básica

In [6]:
# Configuración de estilo para los gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Directorio de resultados
# Para Jupyter Notebook, usar Path.cwd() en lugar de __file__
try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    # Si __file__ no está definido (Jupyter), usar el directorio actual
    SCRIPT_DIR = Path.cwd()

CARPETA_RESULTADOS = SCRIPT_DIR / "resultados_pruebas_masivas"
CARPETA_FUNCIONES_OBJ = SCRIPT_DIR / "resultados" / "funciones_obj"
CARPETA_GRAFICOS = SCRIPT_DIR / "graficos_analisis"
CARPETA_GRAFICOS.mkdir(exist_ok=True)

print(f"Directorio de trabajo: {SCRIPT_DIR}")
print(f"Analizando resultados en: {CARPETA_RESULTADOS}")
print(f"Analizando optimización en: {CARPETA_FUNCIONES_OBJ}")
print(f"Guardando gráficos en: {CARPETA_GRAFICOS}\n")

Directorio de trabajo: c:\Users\romeo\Desktop\TFG_RomeoBernalAlcelay\git\TFG_Romeo
Analizando resultados en: c:\Users\romeo\Desktop\TFG_RomeoBernalAlcelay\git\TFG_Romeo\resultados_pruebas_masivas
Analizando optimización en: c:\Users\romeo\Desktop\TFG_RomeoBernalAlcelay\git\TFG_Romeo\resultados\funciones_obj
Guardando gráficos en: c:\Users\romeo\Desktop\TFG_RomeoBernalAlcelay\git\TFG_Romeo\graficos_analisis



## Funciones auxiliares

In [7]:
def extraer_info_filename(filepath):
    """
    Extrae información del nombre de la carpeta que contiene el CSV.
    """
    folder_name = filepath.parent.name
    parts = folder_name.split('_')
    
    tipo_mapa = "mediano" if parts[0] == "med" else "grande"
    algoritmo = parts[1]
    
    funcion_obj = None
    if algoritmo in ["ACO", "ABC", "BHA"] and len(parts) > 2:
        funcion_obj = parts[2]
    
    return {
        'tipo_mapa': tipo_mapa,
        'algoritmo': algoritmo,
        'funcion_obj': funcion_obj,
        'filename': filepath.name,
        'folder': folder_name
    }

def leer_csv_resultado(filepath):
    """
    Lee un archivo CSV de resultado y extrae las métricas.
    """
    try:
        df = pd.read_csv(filepath, index_col=0)
        
        agent_cols = [col for col in df.columns if col.startswith('Agent')]
        num_agents = len(agent_cols)
        
        metricas = {
            'num_agents': num_agents,
            'distancias': [],
            'pasos': [],
            'exitos': []
        }
        
        for agent_col in agent_cols:
            dist = df.loc['Distance', agent_col]
            if isinstance(dist, str):
                dist = float(dist)
            metricas['distancias'].append(dist)
            
            steps = df.loc['Steps taken', agent_col]
            if isinstance(steps, str):
                steps = int(steps)
            metricas['pasos'].append(steps)
            
            found = df.loc['Found Target', agent_col]
            if isinstance(found, str):
                found = found.strip().lower() == 'true'
            metricas['exitos'].append(1 if found else 0)
        
        return metricas
    
    except Exception as e:
        print(f"Error leyendo {filepath.name}: {e}")
        return None

def leer_csv_optimizacion(filepath):
    """
    Lee archivos CSV de optimización de funciones objetivo.
    Formato: ACO_evolution_max_agents1_iter30_FODTR_20260119_235942.csv
    """
    try:
        # Leer el archivo saltando las líneas de comentario
        df = pd.read_csv(filepath, comment='#')
        
        # Extraer información del nombre del archivo
        filename = filepath.stem
        parts = filename.split('_')
        
        algoritmo = parts[0]  # ACO, ABC, o BHA
        
        # Buscar la función objetivo en el nombre del archivo
        funcion_obj = None
        for part in parts:
            if part.startswith('FO'):
                funcion_obj = part[2:]  # Quitar 'FO' del inicio
                break
        
        return {
            'algoritmo': algoritmo,
            'funcion_obj': funcion_obj,
            'iteraciones': df['Iteración'].values,
            'mejor_obj': df['Mejor Obj'].values,
            'obj_actual': df['Obj Actual'].values,
            'mejor_final': df['Mejor Obj'].iloc[-1]
        }
    
    except Exception as e:
        print(f"Error leyendo optimización {filepath.name}: {e}")
        return None

In [8]:
# ============================================================================
# ANÁLISIS DE RESULTADOS DE SIMULACIÓN
# ============================================================================

archivos_csv = list(CARPETA_RESULTADOS.rglob("*.csv"))
print(f"Archivos CSV de simulación encontrados: {len(archivos_csv)}")

if len(archivos_csv) == 0:
    print("\nNo se encontraron archivos CSV en la carpeta de resultados.")
    print(f"Carpetas disponibles en {CARPETA_RESULTADOS}:")
    for carpeta in sorted(CARPETA_RESULTADOS.iterdir()):
        if carpeta.is_dir():
            num_csv = len(list(carpeta.glob("*.csv")))
            print(f"  - {carpeta.name}: {num_csv} archivos CSV")
else:
    print(f"Estructura detectada:")
    carpetas_unicas = set(csv.parent.name for csv in archivos_csv)
    for carpeta in sorted(carpetas_unicas):
        num = len([c for c in archivos_csv if c.parent.name == carpeta])
        print(f"  - {carpeta}: {num} archivos")
    print()

# Estructura para almacenar datos por algoritmo
datos_por_algoritmo = defaultdict(lambda: {
    'distancias': [],
    'pasos': [],
    'exitos': [],
    'tipo_mapa': [],
    'funcion_obj': []
})

# Procesar cada archivo
for csv_file in archivos_csv:
    info = extraer_info_filename(csv_file)
    metricas = leer_csv_resultado(csv_file)
    
    if metricas is None:
        continue
    
    if info['funcion_obj']:
        clave = f"{info['algoritmo']}_{info['funcion_obj']}"
    else:
        clave = info['algoritmo']
    
    datos_por_algoritmo[clave]['distancias'].append(np.mean(metricas['distancias']))
    datos_por_algoritmo[clave]['pasos'].append(np.mean(metricas['pasos']))
    datos_por_algoritmo[clave]['exitos'].append(np.mean(metricas['exitos']))
    datos_por_algoritmo[clave]['tipo_mapa'].append(info['tipo_mapa'])
    datos_por_algoritmo[clave]['funcion_obj'].append(info['funcion_obj'])

# Crear DataFrame consolidado
data_list = []
for algoritmo, datos in datos_por_algoritmo.items():
    for i in range(len(datos['distancias'])):
        data_list.append({
            'Algoritmo': algoritmo,
            'Distancia': datos['distancias'][i],
            'Pasos': datos['pasos'][i],
            'Éxito': datos['exitos'][i],
            'Tipo_Mapa': datos['tipo_mapa'][i]
        })

df_resultados = pd.DataFrame(data_list)


Archivos CSV de simulación encontrados: 36
Estructura detectada:
  - gde_ABC_DTR_a1_i1: 1 archivos
  - gde_ABC_ET_a1_i1: 1 archivos
  - gde_ABC_ME_a1_i1: 1 archivos
  - gde_ABC_MS_a1_i1: 1 archivos
  - gde_ACO_DTR_a1_i1: 1 archivos
  - gde_ACO_ET_a1_i1: 1 archivos
  - gde_ACO_ME_a1_i1: 1 archivos
  - gde_ACO_MS_a1_i1: 1 archivos
  - gde_BHA_DTR_a1_i1: 1 archivos
  - gde_BHA_ET_a1_i1: 1 archivos
  - gde_BHA_ME_a1_i1: 1 archivos
  - gde_BHA_MS_a1_i1: 1 archivos
  - gde_expanding_sq_a1_i1: 1 archivos
  - gde_lawnmower_a1_i1: 1 archivos
  - gde_voraz-heur_a1_i1: 1 archivos
  - gde_voraz-myope_a1_i1: 1 archivos
  - med_ABC_DTR_a1_i1: 1 archivos
  - med_ABC_ET_a1_i1: 1 archivos
  - med_ABC_ME_a1_i1: 1 archivos
  - med_ABC_MS_a1_i1: 1 archivos
  - med_ACO_DTR_a1_i1: 2 archivos
  - med_ACO_ET_a1_i1: 2 archivos
  - med_ACO_ME_a1_i1: 2 archivos
  - med_ACO_MS_a1_i1: 2 archivos
  - med_BHA_DTR_a1_i1: 1 archivos
  - med_BHA_ET_a1_i1: 1 archivos
  - med_BHA_ME_a1_i1: 1 archivos
  - med_BHA_MS_a1_i1

In [9]:
# ============================================================================
# ANÁLISIS DE OPTIMIZACIÓN DE FUNCIONES OBJETIVO
# ============================================================================

optimizacion_data = []
if CARPETA_FUNCIONES_OBJ.exists():
    archivos_opt = list(CARPETA_FUNCIONES_OBJ.glob("*.csv"))
    print(f"\nArchivos de optimización encontrados: {len(archivos_opt)}")
    
    for csv_file in archivos_opt:
        opt_info = leer_csv_optimizacion(csv_file)
        if opt_info:
            optimizacion_data.append(opt_info)
            print(f"  - {opt_info['algoritmo']}_{opt_info['funcion_obj']}: "
                  f"Mejor={opt_info['mejor_final']:.2e}")
else:
    print(f"\nNo se encontró la carpeta de optimización: {CARPETA_FUNCIONES_OBJ}")

# ============================================================================
# TABLA DE ESTADÍSTICAS GENERALES
# ============================================================================
print("\n" + "="*80)
print("ESTADÍSTICAS POR ALGORITMO")
print("="*80)

estadisticas = df_resultados.groupby('Algoritmo').agg({
    'Distancia': ['mean', 'std', 'min', 'max'],
    'Pasos': ['mean', 'std', 'min', 'max'],
    'Éxito': ['mean', 'count']
}).round(2)

estadisticas.columns = ['_'.join(col).strip() for col in estadisticas.columns.values]
estadisticas.rename(columns={
    'Distancia_mean': 'Dist_Media',
    'Distancia_std': 'Dist_Std',
    'Distancia_min': 'Dist_Min',
    'Distancia_max': 'Dist_Max',
    'Pasos_mean': 'Pasos_Media',
    'Pasos_std': 'Pasos_Std',
    'Pasos_min': 'Pasos_Min',
    'Pasos_max': 'Pasos_Max',
    'Éxito_mean': 'Tasa_Éxito',
    'Éxito_count': 'N_Simulaciones'
}, inplace=True)

print(estadisticas.to_string())
print("\n")

estadisticas.to_csv(CARPETA_GRAFICOS / "estadisticas_generales.csv")


Archivos de optimización encontrados: 4
  - ACO_DTR: Mejor=5.89e-26
  - ACO_MS: Mejor=1.00e+00
  - ACO_ET: Mejor=1.30e+01
  - ACO_ME: Mejor=5.09e+00

ESTADÍSTICAS POR ALGORITMO
             Dist_Media  Dist_Std  Dist_Min  Dist_Max  Pasos_Media  Pasos_Std  Pasos_Min  Pasos_Max  Tasa_Éxito  N_Simulaciones
Algoritmo                                                                                                                      
ABC_DTR          442.06    117.13    359.23    524.88       367.50      95.46      300.0      435.0         0.5               2
ABC_ET            44.49      2.88     42.46     46.53        36.00       1.41       35.0       37.0         1.0               2
ABC_ME           181.44    253.19      2.41    360.48       151.00     210.72        2.0      300.0         0.5               2
ABC_MS           371.48    398.68     89.57    653.39       305.00     328.10       73.0      537.0         1.0               2
ACO_DTR          294.44    280.90    122.30    618.60 

In [10]:
# ============================================================================
# GRÁFICO 1: GRÁFICOS CIRCULARES DE ÉXITO/FRACASO POR ALGORITMO
# ============================================================================
algoritmos_unicos = df_resultados['Algoritmo'].unique()
n_algoritmos = len(algoritmos_unicos)
n_cols = 4
n_rows = int(np.ceil(n_algoritmos / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten() if n_algoritmos > 1 else [axes]

colores_exito = ['#2ecc71', '#e74c3c']  # Verde para éxito, rojo para fracaso

for i, alg in enumerate(sorted(algoritmos_unicos)):
    datos_alg = df_resultados[df_resultados['Algoritmo'] == alg]
    tasa_exito = datos_alg['Éxito'].mean()
    tasa_fracaso = 1 - tasa_exito
    
    if i < len(axes):
        axes[i].pie([tasa_exito, tasa_fracaso], 
                    labels=[f'Éxito\n{tasa_exito*100:.1f}%', f'Fracaso\n{tasa_fracaso*100:.1f}%'],
                    colors=colores_exito,
                    autopct='',
                    startangle=90,
                    textprops={'fontsize': 9, 'weight': 'bold'})
        axes[i].set_title(f'{alg}', fontsize=10, fontweight='bold', pad=10)

# Ocultar ejes sobrantes
for i in range(n_algoritmos, len(axes)):
    axes[i].axis('off')

plt.suptitle('Tasa de Éxito vs Fracaso por Algoritmo', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / 'pie_charts_exito.png', dpi=300, bbox_inches='tight')
plt.close()

In [11]:
# ============================================================================
# GRÁFICO 2: COMPARACIÓN DE DISTANCIA Y PASOS (BARRAS AGRUPADAS)
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distancia media por algoritmo
distancias_media = df_resultados.groupby('Algoritmo')['Distancia'].mean().sort_values()
colores_dist = plt.cm.viridis(np.linspace(0, 1, len(distancias_media)))

distancias_media.plot(kind='barh', ax=axes[0], color=colores_dist)
axes[0].set_xlabel('Distancia Media', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Algoritmo', fontsize=12, fontweight='bold')
axes[0].set_title('Distancia Media Recorrida', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Pasos medios por algoritmo
pasos_media = df_resultados.groupby('Algoritmo')['Pasos'].mean().sort_values()
colores_pasos = plt.cm.plasma(np.linspace(0, 1, len(pasos_media)))

pasos_media.plot(kind='barh', ax=axes[1], color=colores_pasos)
axes[1].set_xlabel('Pasos Medios', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Algoritmo', fontsize=12, fontweight='bold')
axes[1].set_title('Número Medio de Pasos', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / 'barras_distancia_pasos.png', dpi=300, bbox_inches='tight')
plt.close()

In [12]:
# ============================================================================
# GRÁFICO 3: BOXPLOT COMPARATIVO CON COLORES DISTINTIVOS
# ============================================================================
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Crear paleta de colores única para cada algoritmo
n_alg = len(df_resultados['Algoritmo'].unique())
colores_unicos = plt.cm.tab20(np.linspace(0, 1, n_alg))

# Boxplot de distancias
df_sorted_dist = df_resultados.sort_values('Distancia', ascending=False)
algoritmos_orden_dist = df_sorted_dist.groupby('Algoritmo')['Distancia'].median().sort_values(ascending=False).index

sns.boxplot(data=df_resultados, x='Algoritmo', y='Distancia', 
            order=algoritmos_orden_dist, hue='Algoritmo', palette='tab20', legend=False, ax=axes[0])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_title('Distribución de Distancia Recorrida por Algoritmo', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Distancia Recorrida', fontsize=12)
axes[0].set_xlabel('')
axes[0].grid(axis='y', alpha=0.3)

# Boxplot de pasos
algoritmos_orden_pasos = df_resultados.groupby('Algoritmo')['Pasos'].median().sort_values(ascending=False).index

sns.boxplot(data=df_resultados, x='Algoritmo', y='Pasos', 
            order=algoritmos_orden_pasos, hue='Algoritmo', palette='tab20', legend=False, ax=axes[1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_title('Distribución de Pasos Dados por Algoritmo', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Número de Pasos', fontsize=12)
axes[1].set_xlabel('Algoritmo', fontsize=12)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / 'boxplot_comparativo.png', dpi=300, bbox_inches='tight')
plt.close()

C:\Users\romeo\AppData\Local\Temp\ipykernel_14104\449024552.py:16: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
C:\Users\romeo\AppData\Local\Temp\ipykernel_14104\449024552.py:27: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')


In [13]:
# ============================================================================
# GRÁFICO 4: SCATTER PLOT MEJORADO CON ANOTACIONES
# ============================================================================
plt.figure(figsize=(14, 10))

# Crear una paleta de colores distintiva
algoritmos_lista = sorted(df_resultados['Algoritmo'].unique())
colores_scatter = dict(zip(algoritmos_lista, plt.cm.tab20.colors[:len(algoritmos_lista)]))

for alg in algoritmos_lista:
    datos_alg = df_resultados[df_resultados['Algoritmo'] == alg]
    plt.scatter(datos_alg['Pasos'], datos_alg['Distancia'], 
               label=alg, alpha=0.7, s=120, color=colores_scatter[alg],
               edgecolors='black', linewidth=0.5)

plt.xlabel('Número de Pasos', fontsize=13, fontweight='bold')
plt.ylabel('Distancia Recorrida', fontsize=13, fontweight='bold')
plt.title('Relación entre Pasos y Distancia por Algoritmo', fontsize=15, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, framealpha=0.9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / 'scatter_pasos_distancia.png', dpi=300, bbox_inches='tight')
plt.close()

In [14]:
# ============================================================================
# GRÁFICO 5: EVOLUCIÓN DE FUNCIONES OBJETIVO
# ============================================================================
if len(optimizacion_data) > 0:
    funciones_unicas = list(set([opt['funcion_obj'] for opt in optimizacion_data]))
    
    for funcion in funciones_unicas:
        plt.figure(figsize=(12, 7))
        
        datos_funcion = [opt for opt in optimizacion_data if opt['funcion_obj'] == funcion]
        
        for opt in datos_funcion:
            plt.plot(opt['iteraciones'], opt['mejor_obj'], 
                    label=f"{opt['algoritmo']} (Final: {opt['mejor_final']:.2e})",
                    linewidth=2, marker='o', markersize=4, alpha=0.8)
        
        plt.xlabel('Iteración', fontsize=12, fontweight='bold')
        plt.ylabel('Mejor Valor Objetivo', fontsize=12, fontweight='bold')
        plt.title(f'Evolución de Optimización - Función Objetivo: {funcion}', 
                 fontsize=14, fontweight='bold')
        plt.legend(fontsize=10, framealpha=0.9)
        plt.grid(True, alpha=0.3)
        plt.yscale('log')  # Escala logarítmica para valores muy pequeños
        plt.tight_layout()
        plt.savefig(CARPETA_GRAFICOS / f'evolucion_{funcion}.png', dpi=300, bbox_inches='tight')
        plt.close()
    
    # Comparación final de todas las funciones objetivo
    plt.figure(figsize=(12, 7))
    
    resultados_finales = defaultdict(dict)
    for opt in optimizacion_data:
        resultados_finales[opt['algoritmo']][opt['funcion_obj']] = opt['mejor_final']
    
    x = np.arange(len(funciones_unicas))
    width = 0.25
    
    for i, (algoritmo, valores) in enumerate(resultados_finales.items()):
        valores_ordenados = [valores.get(f, 0) for f in funciones_unicas]
        plt.bar(x + i*width, valores_ordenados, width, label=algoritmo, alpha=0.8)
    
    plt.xlabel('Función Objetivo', fontsize=12, fontweight='bold')
    plt.ylabel('Mejor Valor Final (log)', fontsize=12, fontweight='bold')
    plt.title('Comparación de Optimización Final por Función Objetivo', fontsize=14, fontweight='bold')
    plt.xticks(x + width, funciones_unicas)
    plt.legend(fontsize=10)
    plt.yscale('log')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(CARPETA_GRAFICOS / 'comparacion_funciones_objetivo.png', dpi=300, bbox_inches='tight')
    plt.close()

In [15]:
# ============================================================================
# RESUMEN FINAL
# ============================================================================
print("="*80)
print("RESUMEN DEL ANÁLISIS")
print("="*80)
print(f"Total de simulaciones analizadas: {len(df_resultados)}")
print(f"Algoritmos únicos: {df_resultados['Algoritmo'].nunique()}")

algoritmos_base = ['expanding', 'lawnmower', 'voraz-heur', 'voraz-myope']
df_base = df_resultados[df_resultados['Algoritmo'].isin(algoritmos_base)]
df_meta = df_resultados[~df_resultados['Algoritmo'].isin(algoritmos_base)]

print("\n" + "-"*80)
print("MEJORES ALGORITMOS POR MÉTRICA")
print("-"*80)

mejor_dist = df_resultados.groupby('Algoritmo')['Distancia'].mean().idxmin()
valor_dist = df_resultados.groupby('Algoritmo')['Distancia'].mean().min()
print(f"Menor distancia promedio: {mejor_dist} ({valor_dist:.2f} unidades)")

mejor_exito = df_resultados.groupby('Algoritmo')['Éxito'].mean().idxmax()
valor_exito = df_resultados.groupby('Algoritmo')['Éxito'].mean().max()
print(f"Mayor tasa de éxito: {mejor_exito} ({valor_exito*100:.1f}%)")

mejor_pasos = df_resultados.groupby('Algoritmo')['Pasos'].mean().idxmin()
valor_pasos = df_resultados.groupby('Algoritmo')['Pasos'].mean().min()
print(f"Menor número de pasos: {mejor_pasos} ({valor_pasos:.1f} pasos)")

print("\n" + "-"*80)
print("RANKING DE ALGORITMOS (por distancia)")
print("-"*80)
ranking_dist = df_resultados.groupby('Algoritmo').agg({
    'Distancia': 'mean',
    'Pasos': 'mean',
    'Éxito': 'mean'
}).sort_values('Distancia')

for i, (alg, row) in enumerate(ranking_dist.iterrows(), 1):
    print(f"{i:2d}. {alg:15s} | Dist: {row['Distancia']:6.2f} | "
          f"Pasos: {row['Pasos']:6.1f} | Éxito: {row['Éxito']*100:5.1f}%")

print("\n" + "-"*80)
print("COMPARACIÓN: ALGORITMOS BASE vs METAHEURÍSTICOS")
print("-"*80)

if len(df_base) > 0 and len(df_meta) > 0:
    print("\nAlgoritmos BASE:")
    print(f"  Distancia promedio: {df_base['Distancia'].mean():.2f} ± {df_base['Distancia'].std():.2f}")
    print(f"  Pasos promedio: {df_base['Pasos'].mean():.1f} ± {df_base['Pasos'].std():.1f}")
    print(f"  Tasa de éxito: {df_base['Éxito'].mean()*100:.1f}%")
    
    print("\nAlgoritmos METAHEURÍSTICOS:")
    print(f"  Distancia promedio: {df_meta['Distancia'].mean():.2f} ± {df_meta['Distancia'].std():.2f}")
    print(f"  Pasos promedio: {df_meta['Pasos'].mean():.1f} ± {df_meta['Pasos'].std():.1f}")
    print(f"  Tasa de éxito: {df_meta['Éxito'].mean()*100:.1f}%")

print("\n" + "-"*80)
print("ANÁLISIS POR FUNCIÓN OBJETIVO (Metaheurísticos)")
print("-"*80)

funciones_obj = ['DTR', 'ET', 'ME', 'MS']
for funcion in funciones_obj:
    df_func = df_resultados[df_resultados['Algoritmo'].str.contains(funcion, na=False)]
    if len(df_func) > 0:
        print(f"\nFunción {funcion}:")
        print(f"  Algoritmos: {df_func['Algoritmo'].nunique()}")
        print(f"  Distancia promedio: {df_func['Distancia'].mean():.2f} ± {df_func['Distancia'].std():.2f}")
        print(f"  Pasos promedio: {df_func['Pasos'].mean():.1f} ± {df_func['Pasos'].std():.1f}")
        print(f"  Tasa de éxito: {df_func['Éxito'].mean()*100:.1f}%")

if len(optimizacion_data) > 0:
    print("\n" + "-"*80)
    print("RESULTADOS DE OPTIMIZACIÓN DE FUNCIONES OBJETIVO")
    print("-"*80)
    
    for opt in sorted(optimizacion_data, key=lambda x: (x['algoritmo'], x['funcion_obj'])):
        print(f"{opt['algoritmo']}_{opt['funcion_obj']}: Mejor valor final = {opt['mejor_final']:.6e}")

print("\n" + "-"*80)
print(f"Gráficos guardados en: {CARPETA_GRAFICOS}")
print(f"Estadísticas guardadas en: {CARPETA_GRAFICOS / 'estadisticas_generales.csv'}")
print("="*80)

RESUMEN DEL ANÁLISIS
Total de simulaciones analizadas: 36
Algoritmos únicos: 16

--------------------------------------------------------------------------------
MEJORES ALGORITMOS POR MÉTRICA
--------------------------------------------------------------------------------
Menor distancia promedio: ABC_ET (44.49 unidades)
Mayor tasa de éxito: ABC_ET (100.0%)
Menor número de pasos: ABC_ET (36.0 pasos)

--------------------------------------------------------------------------------
RANKING DE ALGORITMOS (por distancia)
--------------------------------------------------------------------------------
 1. ABC_ET          | Dist:  44.49 | Pasos:   36.0 | Éxito: 100.0%
 2. voraz-myope     | Dist:  53.67 | Pasos:   40.0 | Éxito: 100.0%
 3. ACO_ET          | Dist:  77.91 | Pasos:   63.0 | Éxito: 100.0%
 4. ACO_MS          | Dist: 178.93 | Pasos:  148.0 | Éxito: 100.0%
 5. ABC_ME          | Dist: 181.44 | Pasos:  151.0 | Éxito:  50.0%
 6. voraz-heur      | Dist: 199.05 | Pasos:  165.5 | Éxito: 